# Dense Passage Retrieval (DPR) Evaluation Pipeline — Local Indexes & CLI Commands

Uses **already-downloaded local Pyserini indexes** (`C:\Users\Dwija Patel\.cache\pyserini\indexes\`) without re-downloading, and executes `sys.executable` (`{py}`) CLI commands (`pyserini.search.lucene`, `pyserini.eval.convert_trec_run_to_dpr_retrieval_run`, `pyserini.eval.evaluate_dpr_retrieval`) across:
- **All 5 datasets (`nq`, `trivia`, `wq`, `curated`, `squad`)** on the **Multi-dataset index** (`faiss.wikipedia-dpr-100w.dpr_multi.*`) and **BM25** (`lucene-inverted.wikipedia-dpr-100w.*`).
- **NQ only (`nq`)** on the **NQ-only index** (`faiss.wikipedia-dpr-100w.dpr_single-nq.*`).

## 1. Imports & Local Index Auto-Discovery 

In [5]:
import os
import re
import sys
import json
import struct
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer
from pyserini.search import get_topics

py = sys.executable
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RUNS_DIR = Path('runs')
CKPT_DIR = Path('checkpoints')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve your existing local Pyserini index cache directly
INDEX_CACHE_DIR = Path(os.path.expanduser('~')) / '.cache' / 'pyserini' / 'indexes'

def find_local_index(prefix: str, cache_dir: Path = INDEX_CACHE_DIR) -> str:
    matches = sorted([p for p in cache_dir.glob(f'{prefix}*') if p.is_dir()])
    if not matches:
        raise FileNotFoundError(f'Could not find local index matching {prefix}* inside {cache_dir}')
    return str(matches[-1])

root = find_local_index('lucene-inverted.wikipedia-dpr-100w')
faiss_multi_root = find_local_index('faiss.wikipedia-dpr-100w.dpr_multi')
faiss_single_nq_root = find_local_index('faiss.wikipedia-dpr-100w.dpr_single-nq')

print('Python executable (py):', py)
print('Device:', DEVICE)
print('Lucene root:', root)
print('FAISS Multi root:', faiss_multi_root)
print('FAISS Single-NQ root:', faiss_single_nq_root)

Python executable (py): c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe
Device: cpu
Lucene root: C:\Users\Dwija Patel\.cache\pyserini\indexes\lucene-inverted.wikipedia-dpr-100w.20260508.deb4c7b
FAISS Multi root: C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_multi.20200127.f403c3
FAISS Single-NQ root: C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_single-nq.20200115.cd5034


## 2. Dataset & Index Routing Configuration

- **Index built on all datasets (`faiss_multi_root`)** $\rightarrow$ evaluated on **all 5 datasets** (`nq`, `trivia`, `wq`, `curated`, `squad`).
- **Index built only on NQ (`faiss_single_nq_root`)** $\rightarrow$ evaluated **strictly on NQ** (`nq`).

In [6]:
ALL_DATASETS = {
    'nq':      {'topics': 'dpr-nq-test',      'tag': 'nq-test',      'regex': False},
    'trivia':  {'topics': 'dpr-trivia-test',  'tag': 'trivia-test',  'regex': False},
    'wq':      {'topics': 'dpr-wq-test',      'tag': 'wq-test',      'regex': False},
    'curated': {'topics': 'dpr-curated-test', 'tag': 'curated-test', 'regex': True},
    'squad':   {'topics': 'dpr-squad-test',   'tag': 'squad-test',   'regex': False},
}

NQ_ONLY_DATASETS = {
    'nq': ALL_DATASETS['nq']
}

DENSE_CONFIGS = {
    'multi': {
        'local_index_dir': faiss_multi_root,
        'encoder_name': 'facebook/dpr-question_encoder-multiset-base',
        'datasets': ALL_DATASETS,      # All datasets index -> run on ALL datasets (incl. SQuAD)
        'alpha': 1.1,
    },
    'dpr-single-nq': {
        'local_index_dir': faiss_single_nq_root,
        'encoder_name': 'facebook/dpr-question_encoder-single-nq-base',
        'datasets': NQ_ONLY_DATASETS,  # NQ-only index -> run ONLY on NQ
        'alpha': 1.1,
    },
}

TOP_K_RETRIEVE = 1000
CHUNK_SIZE = 50_000

## 3. Exact Pyserini CLI Conversion & Evaluation Helper (`-X utf8`)

In [7]:
def run_cmd(cmd_list):
    print('>>', ' '.join(str(x) for x in cmd_list))
    env = {**os.environ, 'PYTHONUTF8': '1'}
    proc = subprocess.run(cmd_list, capture_output=True, text=True, encoding='utf-8', errors='replace', env=env)
    combined = (proc.stdout or '') + '\n' + (proc.stderr or '')
    if combined.strip():
        print(combined.strip())
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd_list, output=proc.stdout, stderr=proc.stderr)
    return combined


def convert_and_eval_trec(trec_path: Path, topics_name: str, use_regex: bool = False):
    """Runs convert_trec_run_to_dpr_retrieval_run and evaluate_dpr_retrieval with UTF-8 forced on Windows."""
    json_path = trec_path.with_suffix('.json')

    if not json_path.exists():
        run_cmd([
            py, '-X', 'utf8', '-m', 'pyserini.eval.convert_trec_run_to_dpr_retrieval_run',
            '--index', root,
            '--topics', topics_name,
            '--input', str(trec_path),
            '--output', str(json_path),
        ])

    eval_cmd = [
        py, '-X', 'utf8', '-m', 'pyserini.eval.evaluate_dpr_retrieval',
        '--retrieval', str(json_path),
        '--topk', '20', '100',
    ]
    if use_regex:
        eval_cmd.append('--regex')

    out_text = run_cmd(eval_cmd)
    m20 = re.search(r'Top20\s+accuracy:\s*([0-9.]+)', out_text)
    m100 = re.search(r'Top100\s+accuracy:\s*([0-9.]+)', out_text)
    if not m20 or not m100:
        raise RuntimeError(f'Failed to parse Top20/Top100 accuracy from output:\n{out_text}')

    return {
        'Top-20': round(float(m20.group(1)) * 100, 2),
        'Top-100': round(float(m100.group(1)) * 100, 2),
    }

## 4. Run BM25 Across All 5 Datasets Using `pyserini.search.lucene` & Local `root`

In [8]:
bm25_records = []

for ds_key, ds_cfg in ALL_DATASETS.items():
    topics_name = ds_cfg['topics']
    tag = ds_cfg['tag']
    trec_out = RUNS_DIR / f'run.dpr.{tag}.bm25.trec'

    if not trec_out.exists():
        run_cmd([
            py, '-m', 'pyserini.search.lucene',
            '--index', root,
            '--topics', topics_name,
            '--output', str(trec_out),
            '--hits', str(TOP_K_RETRIEVE),
            '--threads', '8',
            '--batch-size', '128',
            '--bm25',
        ])
    else:
        print(f'[Skip] Existing BM25 run found: {trec_out}')

    scores = convert_and_eval_trec(trec_out, topics_name, use_regex=ds_cfg['regex'])
    bm25_records.append({
        'Dataset': ds_key.upper(),
        'Retriever': 'BM25',
        'Index': Path(root).name,
        'Top-20': scores['Top-20'],
        'Top-100': scores['Top-100'],
    })

display(pd.DataFrame(bm25_records))

[Skip] Existing BM25 run found: runs\run.dpr.nq-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.bm25.json --topk 20 100
Top20	accuracy: 0.6294
Top100	accuracy: 0.7825


100%|██████████| 3610/3610 [00:00<00:00, 307965.62it/s]
[Skip] Existing BM25 run found: runs\run.dpr.trivia-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.trivia-test.bm25.json --topk 20 100
Top20	accuracy: 0.7641
Top100	accuracy: 0.8315


100%|██████████| 11313/11313 [00:00<00:00, 265462.87it/s]
[Skip] Existing BM25 run found: runs\run.dpr.wq-test.bm25.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.wq-test.bm25.json --topk 20 100
Top20	accuracy: 0.6240
Top100	accuracy: 0.7549


100%|██████████| 

,Dataset,Retriever,Index,Top-20,Top-100
0,NQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.94,78.25
1,TRIVIA,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,76.41,83.15
2,WQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.40,75.49
3,CURATED,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,80.69,89.91
4,SQUAD,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,71.09,81.84


## 5. Low-RAM `np.memmap` Chunked Exact Inner-Product Search on Local FAISS Indexes

In [9]:
def open_local_faiss_memmap(local_index_dir: str):
    """Memory-maps your local FAISS IndexFlatIP file without loading 64 GB into RAM."""
    index_path = Path(local_index_dir)
    candidates = sorted(index_path.glob('*.faiss')) or [
        p for p in index_path.iterdir() if p.is_file() and p.name not in {'docid', 'README.md'}
    ]
    if not candidates:
        raise FileNotFoundError(f'No FAISS binary found inside {local_index_dir}')
    faiss_file = max(candidates, key=lambda p: p.stat().st_size)

    with open(faiss_file, 'rb') as f:
        fourcc = f.read(4)
        if fourcc != b'IxFI':
            raise ValueError(f'Expected IndexFlatIP (IxFI), got {fourcc} in {faiss_file}')
        dim = struct.unpack('<i', f.read(4))[0]
        ntotal = struct.unpack('<q', f.read(8))[0]
        _dummy1 = struct.unpack('<q', f.read(8))[0]
        _dummy2 = struct.unpack('<q', f.read(8))[0]
        _is_trained = struct.unpack('<?', f.read(1))[0]
        _metric_type = struct.unpack('<i', f.read(4))[0]
        vec_count = struct.unpack('<Q', f.read(8))[0]
        data_offset = f.tell()

    assert vec_count == ntotal * dim, f'Header mismatch: {vec_count} != {ntotal}*{dim}'

    with open(index_path / 'docid', 'r', encoding='utf-8') as f:
        docids = [line.strip() for line in f]
    assert len(docids) == ntotal, f'Docid count ({len(docids)}) != ntotal ({ntotal})'

    vectors = np.memmap(
        faiss_file, dtype=np.float32, mode='r', offset=data_offset, shape=(ntotal, dim)
    )
    print(f'Memory-mapped local index {faiss_file.name}: shape={vectors.shape}')
    return vectors, docids


@torch.no_grad()
def encode_queries(topics_dict: dict, encoder_name: str, batch_size: int = 64):
    tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(encoder_name)
    model = DPRQuestionEncoder.from_pretrained(encoder_name).to(DEVICE)
    model.eval()

    qids = list(topics_dict.keys())
    queries = [topics_dict[qid]['title'] for qid in qids]
    all_embs = []

    for start in range(0, len(queries), batch_size):
        batch_q = queries[start:start + batch_size]
        inputs = tokenizer(
            batch_q, padding=True, truncation=True, max_length=256, return_tensors='pt'
        ).to(DEVICE)
        emb = model(**inputs).pooler_output.cpu().numpy().astype(np.float32)
        all_embs.append(emb)

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return qids, np.vstack(all_embs)


def chunked_exact_ip_search(
    passage_memmap: np.memmap,
    docids: list,
    qids: list,
    query_embs: np.ndarray,
    trec_out_path: Path,
    ckpt_path: Path,
    run_tag: str,
    topk: int = 1000,
    chunk_size: int = 50_000,
    ckpt_every_chunks: int = 20,
):
    if trec_out_path.exists():
        print(f'[Skip] Existing TREC run found: {trec_out_path}')
        return

    ntotal = passage_memmap.shape[0]
    num_q = len(qids)
    start_row = 0

    if ckpt_path.exists():
        ckpt = np.load(ckpt_path)
        start_row = int(ckpt['next_row'])
        best_scores = ckpt['best_scores']
        best_indices = ckpt['best_indices']
        print(f'Resuming {run_tag} from row {start_row:,}/{ntotal:,}')
    else:
        best_scores = np.full((num_q, topk), -np.inf, dtype=np.float32)
        best_indices = np.full((num_q, topk), -1, dtype=np.int64)

    q_T = np.ascontiguousarray(query_embs.T)
    total_chunks = (ntotal - start_row + chunk_size - 1) // chunk_size

    for chunk_idx, row_start in enumerate(
        tqdm(range(start_row, ntotal, chunk_size), total=total_chunks, desc=run_tag), start=1
    ):
        row_end = min(row_start + chunk_size, ntotal)
        chunk = np.asarray(passage_memmap[row_start:row_end], dtype=np.float32)
        scores_T = (chunk @ q_T).T

        k_chunk = min(topk, row_end - row_start)
        part_idx = np.argpartition(-scores_T, kth=k_chunk - 1, axis=1)[:, :k_chunk]
        part_scores = np.take_along_axis(scores_T, part_idx, axis=1)
        global_idx = part_idx + row_start

        merged_scores = np.concatenate([best_scores, part_scores], axis=1)
        merged_indices = np.concatenate([best_indices, global_idx], axis=1)

        top_merge = np.argpartition(-merged_scores, kth=topk - 1, axis=1)[:, :topk]
        best_scores = np.take_along_axis(merged_scores, top_merge, axis=1)
        best_indices = np.take_along_axis(merged_indices, top_merge, axis=1)

        if chunk_idx % ckpt_every_chunks == 0:
            np.savez(
                ckpt_path,
                next_row=np.array(row_end, dtype=np.int64),
                best_scores=best_scores,
                best_indices=best_indices,
            )

    order = np.argsort(-best_scores, axis=1)
    sorted_scores = np.take_along_axis(best_scores, order, axis=1)
    sorted_indices = np.take_along_axis(best_indices, order, axis=1)

    with open(trec_out_path, 'w', encoding='utf-8') as f:
        for qi, qid in enumerate(qids):
            for rank in range(topk):
                doc_idx = int(sorted_indices[qi, rank])
                if doc_idx < 0:
                    continue
                f.write(f'{qid} Q0 {docids[doc_idx]} {rank + 1} {sorted_scores[qi, rank]:.6f} {run_tag}\n')

    if ckpt_path.exists():
        ckpt_path.unlink()
    print(f'[Done] Wrote TREC run: {trec_out_path}')


def load_trec_scores(trec_path: Path):
    runs = {}
    with open(trec_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                qid, _, docid, _, score, _ = parts[:6]
                runs.setdefault(qid, {})[docid] = float(score)
    return runs


def build_hybrid_trec_run(dense_trec: Path, bm25_trec: Path, hybrid_out: Path, alpha: float, run_tag: str, topk: int = 1000):
    if hybrid_out.exists():
        print(f'[Skip] Existing Hybrid run found: {hybrid_out}')
        return
    dense_runs = load_trec_scores(dense_trec)
    bm25_runs = load_trec_scores(bm25_trec)
    with open(hybrid_out, 'w', encoding='utf-8') as f:
        for qid, d_docs in dense_runs.items():
            b_docs = bm25_runs.get(qid, {})
            all_docids = set(d_docs.keys()) | set(b_docs.keys())
            scored = [(docid, d_docs.get(docid, 0.0) + alpha * b_docs.get(docid, 0.0)) for docid in all_docids]
            scored.sort(key=lambda x: x[1], reverse=True)
            for rank, (docid, score) in enumerate(scored[:topk], start=1):
                f.write(f'{qid} Q0 {docid} {rank} {score:.6f} {run_tag}\n')
    print(f'[Done] Wrote Hybrid TREC run: {hybrid_out}')

## 6. Run Dense & Hybrid Retrieval + CLI Evaluation

- **`dpr-multi` (`faiss_multi_root`)** runs across **all 5 datasets** (NQ, TriviaQA, WebQuestions, CuratedTREC, **SQuAD**).
- **`dpr-single-nq` (`faiss_single_nq_root`)** runs **only on NQ** (`dpr-nq-test`).

In [10]:
all_records = list(bm25_records)

for mode_key, cfg in DENSE_CONFIGS.items():
    print(f'\n================ Running {mode_key} on {cfg["local_index_dir"]} ================')
    passage_memmap, docids = open_local_faiss_memmap(cfg['local_index_dir'])

    for ds_key, ds_cfg in cfg['datasets'].items():
        topics_name = ds_cfg['topics']
        tag = ds_cfg['tag']
        topics = get_topics(topics_name)
        qids, query_embs = encode_queries(topics, cfg['encoder_name'])

        dense_trec = RUNS_DIR / f'run.dpr.{tag}.{mode_key}.trec'
        ckpt_file = CKPT_DIR / f'ckpt.dpr.{tag}.{mode_key}.npz'
        chunked_exact_ip_search(
            passage_memmap=passage_memmap,
            docids=docids,
            qids=qids,
            query_embs=query_embs,
            trec_out_path=dense_trec,
            ckpt_path=ckpt_file,
            run_tag=mode_key,
            topk=TOP_K_RETRIEVE,
            chunk_size=CHUNK_SIZE,
        )

        # Evaluate Dense run using your exact CLI commands
        dense_scores = convert_and_eval_trec(dense_trec, topics_name, use_regex=ds_cfg['regex'])
        all_records.append({
            'Dataset': ds_key.upper(),
            'Retriever': 'DPR-Multi' if mode_key == 'multi' else 'DPR-Single (NQ)',
            'Index': Path(cfg['local_index_dir']).name,
            'Top-20': dense_scores['Top-20'],
            'Top-100': dense_scores['Top-100'],
        })

        # Build & evaluate Hybrid (BM25 + Dense) run using your exact CLI commands
        bm25_trec = RUNS_DIR / f'run.dpr.{tag}.bm25.trec'
        hybrid_tag = 'hybrid-multi' if mode_key == 'multi' else 'hybrid-single-nq'
        hybrid_trec = RUNS_DIR / f'run.hybrid.{tag}.{hybrid_tag}.trec'
        build_hybrid_trec_run(
            dense_trec=dense_trec,
            bm25_trec=bm25_trec,
            hybrid_out=hybrid_trec,
            alpha=cfg['alpha'],
            run_tag=hybrid_tag,
            topk=TOP_K_RETRIEVE,
        )

        hybrid_scores = convert_and_eval_trec(hybrid_trec, topics_name, use_regex=ds_cfg['regex'])
        all_records.append({
            'Dataset': ds_key.upper(),
            'Retriever': 'BM25 + DPR-Multi' if mode_key == 'multi' else 'BM25 + DPR-Single (NQ)',
            'Index': Path(cfg['local_index_dir']).name,
            'Top-20': hybrid_scores['Top-20'],
            'Top-100': hybrid_scores['Top-100'],
        })

    del passage_memmap

results_df = pd.DataFrame(all_records)
results_df.to_csv(RUNS_DIR / 'dpr_final_table_results.csv', index=False)
pivot_df = results_df.pivot(index='Retriever', columns='Dataset', values=['Top-20', 'Top-100'])
display(results_df)
display(pivot_df)


================ Running multi on C:\Users\Dwija Patel\.cache\pyserini\indexes\faiss.wikipedia-dpr-100w.dpr_multi.20200127.f403c3 ================
Memory-mapped local index index: shape=(21015324, 768)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Skip] Existing TREC run found: runs\run.dpr.nq-test.multi.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.multi.json --topk 20 100
Top20	accuracy: 0.7947
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 246635.35it/s]
[Skip] Existing Hybrid run found: runs\run.hybrid.nq-test.hybrid-multi.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.hybrid.nq-test.hybrid-multi.json --topk 20 100
Top20	accuracy: 0.8188
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 313558.73it/s]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (18 outputs left out)

dpr-single-nq:   0%|          | 0/421 [00:00<?, ?it/s]

[Done] Wrote TREC run: runs\run.dpr.nq-test.dpr-single-nq.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.convert_trec_run_to_dpr_retrieval_run --index C:\Users\Dwija Patel\.cache\pyserini\indexes\lucene-inverted.wikipedia-dpr-100w.20260508.deb4c7b --topics dpr-nq-test --input runs\run.dpr.nq-test.dpr-single-nq.trec --output runs\run.dpr.nq-test.dpr-single-nq.json
100%|██████████| 3610000/3610000 [19:30<00:00, 3085.37it/s]
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.dpr.nq-test.dpr-single-nq.json --topk 20 100
Top20	accuracy: 0.8006
Top100	accuracy: 0.8609


100%|██████████| 3610/3610 [00:00<00:00, 480619.52it/s]
[Skip] Existing Hybrid run found: runs\run.hybrid.nq-test.hybrid-single-nq.trec
>> c:\Users\Dwija Patel\dwija\sem5\indexing\winenv\Scripts\python.exe -X utf8 -m pyserini.eval.evaluate_dpr_retrieval --retrieval runs\run.hybrid.nq-tes

,Dataset,Retriever,Index,Top-20,Top-100
0,NQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.94,78.25
1,TRIVIA,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,76.41,83.15
2,WQ,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,62.40,75.49
3,CURATED,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,80.69,89.91
4,SQUAD,BM25,lucene-inverted.wikipedia-dpr-100w.20260508.de...,71.09,81.84
5,NQ,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,79.47,86.09
6,NQ,BM25 + DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,81.88,86.09
7,TRIVIA,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,78.87,84.79
8,TRIVIA,BM25 + DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,81.05,84.79
9,WQ,DPR-Multi,faiss.wikipedia-dpr-100w.dpr_multi.20200127.f4...,75.05,82.97


Top-20                             Top-100         \
Dataset                CURATED     NQ  SQUAD TRIVIA     WQ CURATED     NQ   
Retriever                                                                   
BM25                     80.69  62.94  71.09  76.41  62.40   89.91  78.25   
BM25 + DPR-Multi         89.19  81.88  67.28  81.05  76.67   93.37  86.09   
BM25 + DPR-Single (NQ)     NaN  81.86    NaN    NaN    NaN     NaN  86.09   
DPR-Multi                88.76  79.47  51.99  78.87  75.05   93.37  86.09   
DPR-Single (NQ)            NaN  80.06    NaN    NaN    NaN     NaN  86.09   

                                             
Dataset                 SQUAD TRIVIA     WQ  
Retriever                                    
BM25                    81.84  83.15  75.49  
BM25 + DPR-Multi        75.78  84.79  82.97  
BM25 + DPR-Single (NQ)    NaN    NaN    NaN  
DPR-Multi               67.73  84.79  82.97  
DPR-Single (NQ)           NaN    NaN    NaN